# Exercise 5.5
---

In [2]:
#Loading packages
import pandas as pd
import numpy as np
from ISLP import load_data
import statsmodels.api as sm
import statsmodels.formula.api as smf

In [3]:
#Loading data
Default = load_data('Default')
Default.head()

,default,student,balance,income
0,No,No,729.526495,44361.625074
1,No,Yes,817.180407,12106.134700
2,No,No,1073.549164,31767.138947
3,No,No,529.250605,35704.493935
4,No,No,785.655883,38463.495879


## a)
***

In [4]:
#Fit logistic regression model
model_a = smf.glm('default ~ income + balance', data = Default, family = sm.families.Binomial()).fit()
print(model_a.summary())

                        Generalized Linear Model Regression Results                        
Dep. Variable:     ['default[No]', 'default[Yes]']   No. Observations:                10000
Model:                                         GLM   Df Residuals:                     9997
Model Family:                             Binomial   Df Model:                            2
Link Function:                               Logit   Scale:                          1.0000
Method:                                       IRLS   Log-Likelihood:                -789.48
Date:                             Fri, 15 May 2026   Deviance:                       1579.0
Time:                                     19:16:57   Pearson chi2:                 6.95e+03
No. Iterations:                                  9   Pseudo R-squ. (CS):             0.1256
Covariance Type:                         nonrobust                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
-

## b)
***

In [6]:
#Split data 50/50
rng = np.random.default_rng(1)
n = len(Default)
train_idx = rng.choice(n, size = n // 2, replace = False)
val_idx = np.setdiff1d(np.arange(n), train_idx)

train = Default.iloc[train_idx]
val = Default.iloc[val_idx]

In [7]:
#Fit multiple logistic regression model
model_b = smf.glm('default ~ income + balance', data = train, family = sm.families.Binomial()).fit()
print(model_b.summary())

                        Generalized Linear Model Regression Results                        
Dep. Variable:     ['default[No]', 'default[Yes]']   No. Observations:                 5000
Model:                                         GLM   Df Residuals:                     4997
Model Family:                             Binomial   Df Model:                            2
Link Function:                               Logit   Scale:                          1.0000
Method:                                       IRLS   Log-Likelihood:                -377.64
Date:                             Fri, 15 May 2026   Deviance:                       755.28
Time:                                     19:16:58   Pearson chi2:                 4.18e+03
No. Iterations:                                  9   Pseudo R-squ. (CS):             0.1262
Covariance Type:                         nonrobust                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
-

In [8]:
#Predict value of default status
prob = model_b.predict(val)
pred = np.where(prob > 0.5, 'No', 'Yes')

In [14]:
print(prob)
print(pred)

1       0.998852
2       0.992561
3       0.999650
4       0.998376
5       0.998112
          ...   
9995    0.998582
9996    0.999051
9997    0.996567
9998    0.873725
9999    0.999964
Length: 5000, dtype: float64
['No' 'No' 'No' ... 'No' 'No' 'No']


In [10]:
#Validation set error
error_rate = np.mean(pred != val['default'].values)
print(f"Validation error: {error_rate:.4f}")

Validation error: 0.0276


## c)
***

In [11]:
#Repeating three times with different splits
for i in range(3):
    rng2 = np.random.default_rng(i)

    n = len(Default)

    train_idx2 = rng2.choice(n, size = n // 2, replace = False)
    val_idx2 = np.setdiff1d(np.arange(n), train_idx2)

    train2 = Default.iloc[train_idx2]
    val2 = Default.iloc[val_idx2]

    model_c = smf.glm('default ~ income + balance', data = train2, family = sm.families.Binomial()).fit()

    prob2 = model_c.predict(val2)
    pred2 = np.where(prob2 > 0.5, 'No', 'Yes')

    error_rate2 = np.mean(pred2 != val2['default'].values)
    print(f"Seed {i}: Validation error = {error_rate2:.4f}")

Seed 0: Validation error = 0.0270
Seed 1: Validation error = 0.0276
Seed 2: Validation error = 0.0236


### Interpretation

Error rate is between 2.4 - 2.8% depending on which split is used. This illustrates a weakness regarding the validation set approach: estimates is dependant on the randomness of the split. 

## d)
***

In [12]:
#Logistic regression including student as dummy
model_d = smf.glm('default ~ income + balance + student', data = train, family = sm.families.Binomial()).fit()

prob_d = model_d.predict(val)
pred_d = np.where(prob_d > 0.5, 'No', 'Yes')

error_rate_d = np.mean(pred_d != val['default'].values)
print(f"Error rate: {error_rate_d}")

Error rate: 0.028


In [13]:
#Difference
error_rate_diff = error_rate2 - error_rate_d
print(f"Difference in error rate with added variable: {error_rate_diff:.4f}")

Difference in error rate with added variable: -0.0044


### Interpretation

As we can see, the error rate is reduced by 0.0044 points when adding `student` as another variable. This shows that the added variable does not provide additional predictive power, making it redundant.